## Load & Processed

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Cloud

In [2]:
data = pd.read_csv("../cloud/raw_data/cloud_multimodal_qa_raw.csv")

In [3]:
data.shape

(45796, 11)

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45796 entries, 0 to 45795
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   title            45796 non-null  object
 1   body             45796 non-null  object
 2   tags             45796 non-null  object
 3   link             45796 non-null  object
 4   score            45796 non-null  int64 
 5   creation_date    45796 non-null  object
 6   answer_count     45796 non-null  int64 
 7   comments         45796 non-null  object
 8   answers          45796 non-null  object
 9   selected_answer  45796 non-null  object
 10  images           45796 non-null  object
dtypes: int64(2), object(9)
memory usage: 3.8+ MB


In [5]:
data['title'].duplicated().sum()

np.int64(3149)

In [6]:
data = data.drop_duplicates(subset=['title'], keep='first').reset_index(drop=True)


In [7]:
data['title'].isna().sum()

np.int64(0)

In [8]:
data.shape

(42647, 11)

In [9]:
data.head(2)

,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images
0,Creating Azure monitor alert rule using REST API,\n \nI am trying to create an A...,"['azure', 'rest', 'azure-monitoring', 'azure-m...",https://stackoverflow.com/questions/77900171/c...,0,2024-01-29 13:45:43Z,1,[],[{'body': '\n \nWhen I ran belo...,\n \nWhen I ran below API call ...,"['https://i.imgur.com/NwVqGBF.png', 'https://i..."
1,azure workbook - dropdown parameter values dep...,\n \nI want to have two paramet...,"['azure', 'kql', 'azure-monitor', 'azure-monit...",https://stackoverflow.com/questions/77143230/a...,0,2023-09-20 14:01:57Z,1,"['Provide input and output, how it looks(provi...",[{'body': '\nTo have one parameter depend on a...,"\nTo have one parameter depend on another, you...","['https://i.sstatic.net/y3Iiq.png', 'https://i..."


In [10]:
data.head(1)

,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images
0,Creating Azure monitor alert rule using REST API,\n \nI am trying to create an A...,"['azure', 'rest', 'azure-monitoring', 'azure-m...",https://stackoverflow.com/questions/77900171/c...,0,2024-01-29 13:45:43Z,1,[],[{'body': '\n \nWhen I ran belo...,\n \nWhen I ran below API call ...,"['https://i.imgur.com/NwVqGBF.png', 'https://i..."


In [11]:
data['answers'][321]

'[{\'body\': \'\\nYou need to specify at least any one of the rule destination like CIDR block, a security group ID or a prefix list.\\nBelow code snippet works for you. I have used cidr_blocks in this case.\\nresource "aws_security_group" "public-instance" {\\n  vpc_id      = aws_vpc.study.id\\n  name        = "public-instance"\\n  description = "Group for public instance"\\n\\n  ingress {\\n    description = "Port 80 ingress"\\n    from_port   = 80\\n    to_port     = 80\\n    protocol    = "tcp"\\n    cidr_blocks = ["0.0.0.0/0"]\\n  }\\n\\n  ingress {\\n    description = "Port 22 ingress"\\n    from_port   = 22\\n    to_port     = 22\\n    protocol    = "tcp"\\n    cidr_blocks = ["0.0.0.0/0"]\\n  }\\n\\n  egress {\\n    from_port   = 0\\n    to_port     = 0\\n    protocol    = "all"\\n    cidr_blocks = ["0.0.0.0/0"]\\n  }\\n}\\n\\n    \', \'score\': \'\\n2        \'}, {\'body\': \'\\nAdd cidr_blocks = ["<your ip cidr>"] and change protocol = "tcp"\\n  ingress {\\n    from_port   = 8

## procssed

### answer selection

In [12]:
data[data['answers'].apply(lambda x: len(x) != 0) & data['selected_answer'].isnull()]


,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images


In [13]:
data['answers'][0]

'[{\'body\': \'\\n                \\nWhen I ran below API call with your request body in my environment, I too got same error:\\nPUT https://management.azure.com/subscriptions/subid/resourcegroups/rgname/providers/Microsoft.Insights/alertrules/rulename?api-version=2016-03-01\\n\\n{\\n "location": "westus",\\n "properties": {\\n  "description": "Test Alert",\\n  "isEnabled": "true",\\n  "severity": 2,\\n  "evaluationFrequency": "PT1M",\\n  "windowSize": "PT5M",\\n  "condition": {\\n   "odata.type": "Microsoft.Azure.Monitor.SingleResourceMultipleMetricCriteria",\\n   "allOf": [\\n    {\\n     "name": "metric1",\\n     "metricNamespace": "Microsoft.CognitiveServices/accounts",\\n     "metricName": "TokenTransaction",\\n     "operator": "GreaterThan",\\n     "threshold": 50000,\\n     "timeAggregation": "Average",\\n     "dimensions": [\\n       {\\n         "name": "FeatureName",\\n         "operator": "Include",\\n         "values": [\\n           "gpt-4"\\n         ]\\n       }\\n     ]

In [14]:
'''
data[
    data['answers'].apply(
        lambda x: len(x) != 0 and any(int(str(ans['score']).strip()) > 1 for ans in x if ans.get('score') is not None)
    ) & data['selected_answer'].isnull()
]
'''

"\ndata[\n    data['answers'].apply(\n        lambda x: len(x) != 0 and any(int(str(ans['score']).strip()) > 1 for ans in x if ans.get('score') is not None)\n    ) & data['selected_answer'].isnull()\n]\n"

In [15]:
# checking score
'''
data[
    data['answers'].apply(lambda x: len(x) != 0 and any(int(ans['score'].strip()) > 1 for ans in x))
    & data['selected_answer'].isnull()
]

'''

"\ndata[\n    data['answers'].apply(lambda x: len(x) != 0 and any(int(ans['score'].strip()) > 1 for ans in x))\n    & data['selected_answer'].isnull()\n]\n\n"

In [16]:
'''
def get_best_answer_if_missing(row):
    if pd.isnull(row['selected_answer']):
        # Robustly handle scores that might be integers or strings
        high_score_answers = [
            ans for ans in row['answers']
            if ans.get('score') is not None and int(str(ans['score']).strip()) > 1
        ]
        if high_score_answers:
            # Select the answer with the highest score
            best_answer = max(
                high_score_answers,
                key=lambda x: int(str(x['score']).strip())
            )
            return best_answer['body'].strip()
    return row['selected_answer']

data['selected_answer'] = data.apply(get_best_answer_if_missing, axis=1)
'''

"\ndef get_best_answer_if_missing(row):\n    if pd.isnull(row['selected_answer']):\n        # Robustly handle scores that might be integers or strings\n        high_score_answers = [\n            ans for ans in row['answers']\n            if ans.get('score') is not None and int(str(ans['score']).strip()) > 1\n        ]\n        if high_score_answers:\n            # Select the answer with the highest score\n            best_answer = max(\n                high_score_answers,\n                key=lambda x: int(str(x['score']).strip())\n            )\n            return best_answer['body'].strip()\n    return row['selected_answer']\n\ndata['selected_answer'] = data.apply(get_best_answer_if_missing, axis=1)\n"

In [17]:
# filter
def get_best_answer_if_missing(row):
    if pd.isnull(row['selected_answer']):
        high_score_answers = [ans for ans in row['answers'] if int(ans['score'].strip()) >= 2]
        if high_score_answers:
            # take highest score answer
            best_answer = max(high_score_answers, key=lambda x: int(x['score'].strip()))
            return best_answer['body'].strip()
    return row['selected_answer']

data['selected_answer'] = data.apply(get_best_answer_if_missing, axis=1)

In [18]:
data = data[data['answers'].apply(lambda x: len(x) != 0)].reset_index(drop=True)

In [19]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42647 entries, 0 to 42646
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   title            42647 non-null  object
 1   body             42647 non-null  object
 2   tags             42647 non-null  object
 3   link             42647 non-null  object
 4   score            42647 non-null  int64 
 5   creation_date    42647 non-null  object
 6   answer_count     42647 non-null  int64 
 7   comments         42647 non-null  object
 8   answers          42647 non-null  object
 9   selected_answer  42647 non-null  object
 10  images           42647 non-null  object
dtypes: int64(2), object(9)
memory usage: 3.6+ MB


In [20]:
data = data[data['selected_answer'].notnull()].reset_index(drop=True)


In [21]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42647 entries, 0 to 42646
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   title            42647 non-null  object
 1   body             42647 non-null  object
 2   tags             42647 non-null  object
 3   link             42647 non-null  object
 4   score            42647 non-null  int64 
 5   creation_date    42647 non-null  object
 6   answer_count     42647 non-null  int64 
 7   comments         42647 non-null  object
 8   answers          42647 non-null  object
 9   selected_answer  42647 non-null  object
 10  images           42647 non-null  object
dtypes: int64(2), object(9)
memory usage: 3.6+ MB


In [22]:
data.head(2)

,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images
0,Creating Azure monitor alert rule using REST API,\n \nI am trying to create an A...,"['azure', 'rest', 'azure-monitoring', 'azure-m...",https://stackoverflow.com/questions/77900171/c...,0,2024-01-29 13:45:43Z,1,[],[{'body': '\n \nWhen I ran belo...,\n \nWhen I ran below API call ...,"['https://i.imgur.com/NwVqGBF.png', 'https://i..."
1,azure workbook - dropdown parameter values dep...,\n \nI want to have two paramet...,"['azure', 'kql', 'azure-monitor', 'azure-monit...",https://stackoverflow.com/questions/77143230/a...,0,2023-09-20 14:01:57Z,1,"['Provide input and output, how it looks(provi...",[{'body': '\nTo have one parameter depend on a...,"\nTo have one parameter depend on another, you...","['https://i.sstatic.net/y3Iiq.png', 'https://i..."


In [23]:
# multimodal filter
multimodal_qa = data[data['images'].apply(lambda x: len(x) != 0)].reset_index(drop=True)

In [24]:
multimodal_qa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42647 entries, 0 to 42646
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   title            42647 non-null  object
 1   body             42647 non-null  object
 2   tags             42647 non-null  object
 3   link             42647 non-null  object
 4   score            42647 non-null  int64 
 5   creation_date    42647 non-null  object
 6   answer_count     42647 non-null  int64 
 7   comments         42647 non-null  object
 8   answers          42647 non-null  object
 9   selected_answer  42647 non-null  object
 10  images           42647 non-null  object
dtypes: int64(2), object(9)
memory usage: 3.6+ MB


In [25]:
multimodal_qa['images'][0]

"['https://i.imgur.com/NwVqGBF.png', 'https://i.imgur.com/scgMmwx.png', 'https://i.imgur.com/jGDXJnL.png']"

In [26]:
import ast

def row_passes_filter(row):
    try:
        if int(row['score']) > 2:
            answers = row['answers']
            if isinstance(answers, str):
                answers = ast.literal_eval(answers)

            selected_body = str(row.get('selected_answer', '')).strip()

            for ans in answers:
                answer_score = int(str(ans.get('score')).strip())
                answer_body = str(ans.get('body')).strip()

                if selected_body == answer_body and answer_score > 2:
                    return True  # Keep this row
        return False  # Filter out if condition not met
    except:
        return False  # Filter out on error

multimodal_qa= multimodal_qa[multimodal_qa.apply(row_passes_filter, axis=1)]

In [27]:
multimodal_qa.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8044 entries, 5 to 42640
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   title            8044 non-null   object
 1   body             8044 non-null   object
 2   tags             8044 non-null   object
 3   link             8044 non-null   object
 4   score            8044 non-null   int64 
 5   creation_date    8044 non-null   object
 6   answer_count     8044 non-null   int64 
 7   comments         8044 non-null   object
 8   answers          8044 non-null   object
 9   selected_answer  8044 non-null   object
 10  images           8044 non-null   object
dtypes: int64(2), object(9)
memory usage: 754.1+ KB


In [28]:
import ast
import pandas as pd

def extract_answer_scores(x):
    if pd.isna(x) or x == "":
        return []
    try:
        items = ast.literal_eval(x)  # expects: list of dicts
        scores = []
        for d in items:
            s = d.get("score", None)
            if s is None:
                continue
            # score is stored like "\n104        " -> strip and int
            s = str(s).strip()
            if s != "":
                scores.append(int(s))
        return scores
    except Exception:
        return []

multimodal_qa["ans_score_ins"] = multimodal_qa["answers"].apply(extract_answer_scores)


In [29]:
multimodal_qa['ans_score_ins'].value_counts()

ans_score_ins
[3]                     538
[4]                     439
[5]                     369
[6]                     260
[7]                     186
                       ... 
[7, 7, 6, 6, 1, 1]        1
[9, 15, 9, 0]             1
[16, 4, -1]               1
[68, 5]                   1
[16, 14, 3, 1, 1, 0]      1
Name: count, Length: 2746, dtype: int64

In [30]:
# v4 - final - after manually preprocessing

import pandas as pd

cloud_train_v4 = pd.read_csv("../data/cloud/filter-data/cloud_train.csv")
cloud_val_v4   = pd.read_csv("../data/cloud/filter-data/cloud_val.csv")
cloud_test_v4  = pd.read_csv("../data/cloud/filter-data/cloud_test.csv")

print("train:", cloud_train_v4.shape)
print("val:",   cloud_val_v4.shape)
print("test:",  cloud_test_v4.shape)

cloud_v4_merged = pd.concat(
    [cloud_train_v4, cloud_val_v4, cloud_test_v4],
    axis=0,
    ignore_index=True
)

print("merged:", cloud_v4_merged.shape)

train: (4693, 13)
val: (1484, 13)
test: (1568, 13)
merged: (7745, 13)


In [32]:
# Create clean titles (with raw string to avoid warnings)
cloud_v4_merged['title_clean'] = cloud_v4_merged['title'].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
multimodal_qa['title_clean'] = multimodal_qa['title'].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)

# Get the rows in cloud_v4_merged whose cleaned title exists in multimodal_qa
matched_mask = cloud_v4_merged['title_clean'].isin(multimodal_qa['title_clean'])
print(matched_mask.value_counts())
# Should show: True 7745, False ...

# Filtered dataframe with only matching titles
cloud_v4_merged_filtered = cloud_v4_merged[matched_mask].copy()

print(f"Filtered cloud_v4_merged shape: {cloud_v4_merged_filtered.shape}")

title_clean
True    7745
Name: count, dtype: int64
Filtered cloud_v4_merged shape: (7745, 14)


# device

In [33]:
data = pd.read_csv("../device/raw_data/device_multimodal_qa_raw.csv")

In [34]:
data.shape

(71359, 11)

In [35]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71359 entries, 0 to 71358
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   title            71359 non-null  object
 1   body             71359 non-null  object
 2   tags             71359 non-null  object
 3   link             71359 non-null  object
 4   score            71359 non-null  int64 
 5   creation_date    71359 non-null  object
 6   answer_count     71359 non-null  int64 
 7   comments         71359 non-null  object
 8   answers          71359 non-null  object
 9   selected_answer  71359 non-null  object
 10  images           71359 non-null  object
dtypes: int64(2), object(9)
memory usage: 6.0+ MB


In [36]:
data['title'].duplicated().sum()

np.int64(64042)

In [37]:
data = data.drop_duplicates(subset=['title'], keep='first').reset_index(drop=True)


In [38]:
data['title'].isna().sum()

np.int64(0)

In [39]:
data.shape

(7317, 11)

In [40]:
data.head(2)

,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images
0,Metasploit search features returns intersectio...,\n \nWhen I use multiple search...,"['exploit', 'metasploit', 'exploit', 'metasplo...",https://security.stackexchange.com/questions/2...,1,2024-01-30 01:32:22Z,1,['I appreciate the lengthy explanation. Using ...,[{'body': '\nUse keywords for your search; oth...,"\nUse keywords for your search; otherwise, Met...",['https://i.sstatic.net/0Ukoc.png']
1,Using Metasploit on Kali to attack a Linux mac...,\n \n \n ...,"['metasploit', 'meterpreter', 'metasploit', 'm...",https://security.stackexchange.com/questions/2...,0,2024-02-07 00:15:01Z,1,['Please do not post screenshots of your entir...,[{'body': '\nYou say you are trying to attack ...,You say you are trying to attack a linux machi...,"['https://i.sstatic.net/mSEof.jpg', 'https://i..."


In [41]:
data.head(1)

,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images
0,Metasploit search features returns intersectio...,\n \nWhen I use multiple search...,"['exploit', 'metasploit', 'exploit', 'metasplo...",https://security.stackexchange.com/questions/2...,1,2024-01-30 01:32:22Z,1,['I appreciate the lengthy explanation. Using ...,[{'body': '\nUse keywords for your search; oth...,"\nUse keywords for your search; otherwise, Met...",['https://i.sstatic.net/0Ukoc.png']


In [42]:
data['answers'][321]

'[{\'body\': "\\nWhen the ICMP echo requests all enter the tunnel (check on the pfSense) and no replies return, it\'s either the tunnel that\'s not working (unlikely, you\'d see some symptoms on your side) or the far side isn\'t working.\\n\\nOn the far side, it can either be a problem with the ovpn gateway or with routing back into the tunnel. A problem with routing back might be visible using traceroute, depending on how far beyond the tunnel end the destination is and where the routing stops working.\\n\\nSince you\'re having this problem only when you ping from a certain subnet(?) my bet is on a routing problem.\\n    ", \'score\': \'\\n0        \'}, {\'body\': \'\\nFound the issue!\\n\\nThe tunnel I was troubleshooting was set up as a vpn client (from the firewall perspective). The was also a vpn server set up.\\n\\nWhen dumping on the vpn-interface rather than the lan-interface the problem showed its face more clearly:\\n\\n# tcpdump -i ovpnc1 -n host 172.16.66.2 and icmp\\ntcpdu

## procssed

### answer selection

In [43]:
data[data['answers'].apply(lambda x: len(x) != 0) & data['selected_answer'].isnull()]


,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images


In [44]:
data['answers'][0]

'[{\'body\': \'\\nUse keywords for your search; otherwise, Metasploit searches for your terms in all fields.\\nUsing your search as an example, it returns 255 hits (results truncated):\\n\\nThen, using keywords:\\nmsf6 > search  platform:linux  name:http description:"remote code execution"\\n\\nMatching Modules\\n================\\n\\n   #  Name                                         Disclosure Date  Rank       Check  Description\\n   -  ----                                         ---------------  ----       -----  -----------\\n   0  exploit/linux/misc/jenkins_ldap_deserialize  2016-11-16       excellent  Yes    Jenkins CLI HTTP Java Deserialization Vulnerability\\n   1  exploit/linux/http/ueb_api_rce               2017-08-08       excellent  Yes    Unitrends UEB http api remote code execution\\n\\n\\nYou\\\'ll need to play with it a bit.  I had to use \\\'remote code execution\\\' with the description keyword because \\\'rce\\\' didn\\\'t return any hits.  The Jenkins exploit is in

In [45]:
'''
data[
    data['answers'].apply(
        lambda x: len(x) != 0 and any(int(str(ans['score']).strip()) > 1 for ans in x if ans.get('score') is not None)
    ) & data['selected_answer'].isnull()
]
'''

"\ndata[\n    data['answers'].apply(\n        lambda x: len(x) != 0 and any(int(str(ans['score']).strip()) > 1 for ans in x if ans.get('score') is not None)\n    ) & data['selected_answer'].isnull()\n]\n"

In [46]:
# checking score
'''
data[
    data['answers'].apply(lambda x: len(x) != 0 and any(int(ans['score'].strip()) > 1 for ans in x))
    & data['selected_answer'].isnull()
]

'''

"\ndata[\n    data['answers'].apply(lambda x: len(x) != 0 and any(int(ans['score'].strip()) > 1 for ans in x))\n    & data['selected_answer'].isnull()\n]\n\n"

In [47]:
'''
def get_best_answer_if_missing(row):
    if pd.isnull(row['selected_answer']):
        # Robustly handle scores that might be integers or strings
        high_score_answers = [
            ans for ans in row['answers']
            if ans.get('score') is not None and int(str(ans['score']).strip()) > 1
        ]
        if high_score_answers:
            # Select the answer with the highest score
            best_answer = max(
                high_score_answers,
                key=lambda x: int(str(x['score']).strip())
            )
            return best_answer['body'].strip()
    return row['selected_answer']

data['selected_answer'] = data.apply(get_best_answer_if_missing, axis=1)
'''

"\ndef get_best_answer_if_missing(row):\n    if pd.isnull(row['selected_answer']):\n        # Robustly handle scores that might be integers or strings\n        high_score_answers = [\n            ans for ans in row['answers']\n            if ans.get('score') is not None and int(str(ans['score']).strip()) > 1\n        ]\n        if high_score_answers:\n            # Select the answer with the highest score\n            best_answer = max(\n                high_score_answers,\n                key=lambda x: int(str(x['score']).strip())\n            )\n            return best_answer['body'].strip()\n    return row['selected_answer']\n\ndata['selected_answer'] = data.apply(get_best_answer_if_missing, axis=1)\n"

In [48]:
# filter
def get_best_answer_if_missing(row):
    if pd.isnull(row['selected_answer']):
        high_score_answers = [ans for ans in row['answers'] if int(ans['score'].strip()) > 0]
        if high_score_answers:
            # take highest score answer
            best_answer = max(high_score_answers, key=lambda x: int(x['score'].strip()))
            return best_answer['body'].strip()
    return row['selected_answer']

data['selected_answer'] = data.apply(get_best_answer_if_missing, axis=1)

In [49]:
data = data[data['answers'].apply(lambda x: len(x) != 0)].reset_index(drop=True)

In [50]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7317 entries, 0 to 7316
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   title            7317 non-null   object
 1   body             7317 non-null   object
 2   tags             7317 non-null   object
 3   link             7317 non-null   object
 4   score            7317 non-null   int64 
 5   creation_date    7317 non-null   object
 6   answer_count     7317 non-null   int64 
 7   comments         7317 non-null   object
 8   answers          7317 non-null   object
 9   selected_answer  7317 non-null   object
 10  images           7317 non-null   object
dtypes: int64(2), object(9)
memory usage: 628.9+ KB


In [51]:
data = data[data['selected_answer'].notnull()].reset_index(drop=True)


In [52]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7317 entries, 0 to 7316
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   title            7317 non-null   object
 1   body             7317 non-null   object
 2   tags             7317 non-null   object
 3   link             7317 non-null   object
 4   score            7317 non-null   int64 
 5   creation_date    7317 non-null   object
 6   answer_count     7317 non-null   int64 
 7   comments         7317 non-null   object
 8   answers          7317 non-null   object
 9   selected_answer  7317 non-null   object
 10  images           7317 non-null   object
dtypes: int64(2), object(9)
memory usage: 628.9+ KB


In [53]:
data.head(2)

,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images
0,Metasploit search features returns intersectio...,\n \nWhen I use multiple search...,"['exploit', 'metasploit', 'exploit', 'metasplo...",https://security.stackexchange.com/questions/2...,1,2024-01-30 01:32:22Z,1,['I appreciate the lengthy explanation. Using ...,[{'body': '\nUse keywords for your search; oth...,"\nUse keywords for your search; otherwise, Met...",['https://i.sstatic.net/0Ukoc.png']
1,Using Metasploit on Kali to attack a Linux mac...,\n \n \n ...,"['metasploit', 'meterpreter', 'metasploit', 'm...",https://security.stackexchange.com/questions/2...,0,2024-02-07 00:15:01Z,1,['Please do not post screenshots of your entir...,[{'body': '\nYou say you are trying to attack ...,You say you are trying to attack a linux machi...,"['https://i.sstatic.net/mSEof.jpg', 'https://i..."


In [54]:
# multimodal filter
multimodal_qa = data[data['images'].apply(lambda x: len(x) != 0)].reset_index(drop=True)

In [55]:
multimodal_qa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7317 entries, 0 to 7316
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   title            7317 non-null   object
 1   body             7317 non-null   object
 2   tags             7317 non-null   object
 3   link             7317 non-null   object
 4   score            7317 non-null   int64 
 5   creation_date    7317 non-null   object
 6   answer_count     7317 non-null   int64 
 7   comments         7317 non-null   object
 8   answers          7317 non-null   object
 9   selected_answer  7317 non-null   object
 10  images           7317 non-null   object
dtypes: int64(2), object(9)
memory usage: 628.9+ KB


In [56]:
multimodal_qa['images'][0]

"['https://i.sstatic.net/0Ukoc.png']"

In [57]:
import ast

def row_passes_filter(row):
    try:
        if int(row['score']) > 0:
            answers = row['answers']
            if isinstance(answers, str):
                answers = ast.literal_eval(answers)

            selected_body = str(row.get('selected_answer', '')).strip()

            for ans in answers:
                answer_score = int(str(ans.get('score')).strip())
                answer_body = str(ans.get('body')).strip()

                if selected_body == answer_body and answer_score > 0:
                    return True  # Keep this row
        return False  # Filter out if condition not met
    except:
        return False  # Filter out on error

multimodal_qa= multimodal_qa[data.apply(row_passes_filter, axis=1)]

In [58]:
multimodal_qa.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4715 entries, 2 to 7316
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   title            4715 non-null   object
 1   body             4715 non-null   object
 2   tags             4715 non-null   object
 3   link             4715 non-null   object
 4   score            4715 non-null   int64 
 5   creation_date    4715 non-null   object
 6   answer_count     4715 non-null   int64 
 7   comments         4715 non-null   object
 8   answers          4715 non-null   object
 9   selected_answer  4715 non-null   object
 10  images           4715 non-null   object
dtypes: int64(2), object(9)
memory usage: 442.0+ KB


In [59]:
import ast
import pandas as pd

def extract_answer_scores(x):
    if pd.isna(x) or x == "":
        return []
    try:
        items = ast.literal_eval(x)  # expects: list of dicts
        scores = []
        for d in items:
            s = d.get("score", None)
            if s is None:
                continue
            # score is stored like "\n104        " -> strip and int
            s = str(s).strip()
            if s != "":
                scores.append(int(s))
        return scores
    except Exception:
        return []

multimodal_qa["ans_score_ins"] = multimodal_qa["answers"].apply(extract_answer_scores)


In [60]:
multimodal_qa['ans_score_ins'].value_counts()

ans_score_ins
[2]                                              592
[1]                                              487
[3]                                              343
[4]                                              186
[2, 0]                                           129
                                                ... 
[3, 3, 1, 1, 1, 0]                                 1
[9, 4, 2, 1, -3]                                   1
[380, 102, 76, 45, 19, 18, 16, 4, 3, 1, 0, 0]      1
[16, 10, 4, 4, 1, 0, 0, 0]                         1
[323, 32, 21, 15, 4, 3, 1, 1]                      1
Name: count, Length: 1336, dtype: int64

In [ ]:
data['body'].duplicated().sum()

np.int64(2)

In [62]:
# v4 - final - after manually preprocessing

import pandas as pd

device_train_v4 = pd.read_csv("../data/device/filter-data/device_train.csv")
device_val_v4   = pd.read_csv("../data/device/filter-data/device_val.csv")
device_test_v4  = pd.read_csv("../data/device/filter-data/device_test.csv")

print("train:", device_train_v4.shape)
print("val:",   device_val_v4.shape)
print("test:",  device_test_v4.shape)

device_v4_merged = pd.concat(
    [device_train_v4, device_val_v4, device_test_v4],
    axis=0,
    ignore_index=True
)

print("merged:", device_v4_merged.shape)

train: (2942, 12)
val: (991, 12)
test: (947, 12)
merged: (4880, 12)


In [65]:
# Create clean titles (with raw string to avoid warnings)
device_v4_merged['title_clean'] = device_v4_merged['title'].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
multimodal_qa['title_clean'] = multimodal_qa['title'].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)

# Get the rows in cloud_v4_merged whose cleaned title exists in multimodal_qa
matched_mask = device_v4_merged['title_clean'].isin(multimodal_qa['title_clean'])
print(matched_mask.value_counts())
# Should show: True 7745, False ...

# Filtered dataframe with only matching titles
device_v4_merged_filtered = device_v4_merged[matched_mask].copy()

print(f"Filtered cloud_v4_merged shape: {device_v4_merged_filtered.shape}")

title_clean
True     3827
False    1053
Name: count, dtype: int64
Filtered cloud_v4_merged shape: (3827, 13)
